# 18 · Buffer study area, checked against shapely

## What this checks

**Buffer study area** widened a study area's boundary by a stated distance. This check covers release 0.1.0. The
widget is now retired. The check stays as the record of what that release computed.

The widened boundary was not used to count anything. It set how far past the study area to fetch a street network or
a raster, so that routes near the edge were not cut short.

## How it is checked

**This widget cannot agree exactly with a second program**, and this check says so. Fieldwork buffered with Turf, a
JavaScript geometry library. Turf draws on a flat map centred on the shape, where distances from the centre are true.
The reference buffers the same boundary on a different flat map, UTM zone 30 north, a grid in metres. Two different
maps cannot give identical shapes. So the check compares areas, and allows a small relative difference.

Both sides draw each rounded corner with the same number of steps: 8 in each quarter circle.

The reference uses **shapely**, the Python library for shapes, and **pyproj**, the Python link to the PROJ projection
library. Shapely uses GEOS, the geometry library behind most GIS software. No code is shared with Fieldwork.

The data are the John Snow example's study area in Soho, buffered by 500 m.

## What should happen

1. **The area** is within 0.5% of the reference buffer, drawn with the same 8 steps in each quarter.
2. **The original is inside.** The widened boundary contains the original one, as a buffer must.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["pyproj", "shapely", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_18_buffer_area import run

report = run(Path.cwd() / "fixtures" / "buffer-area-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"])
print(f"  source boundary          {report['sourceAreaSquareMetres']:12.0f} m2")
print(f"  buffered by {report['distanceMetres']} m        {report['fieldworkAreaSquareMetres']:12.0f} m2")
print(f"  {report['referenceCRS']} reference     {report['referenceAreaSquareMetres']:12.0f} m2")
print(f"  relative difference      {report['relativeAreaDifference']:12.3e}")
print(f"  contains the original    {report['containsTheOriginal']}")

## The result on record

First recorded on 11 October 2026, against release 0.1.0. Rerun on 26 November 2026 on the same data: **Agrees**.
The widget is retired, so Fieldwork can no longer export its data again; the rerun repeats the check, not the
export.

Fieldwork's buffered area was 12,248,049 square metres. The reference gave 12,235,014 square metres. That is a
relative difference of about 1.1 × 10⁻³, or 0.11%, well inside the 0.5% allowed. The widened boundary contains the
original.

Running the cell above repeats the check on the same data.

## Why 8 segments and not 32

**Geometry buffer** uses 32 steps in each quarter. This widget used 8. That is not inconsistent. It follows from what
each edge is for:

- **A scope's edge decides what is inside.** So Geometry buffer pays for the finer setting.
- **This boundary only set how far to fetch data.** Its edge decided nothing.

The exact formula for a polygon inside a circle shows what each setting costs. At 8 steps, 0.64% of a true circle
lies outside the polygon, which has 32 corners. At 32 steps, with 128 corners, 0.04% does.

In [ ]:
import math

def fraction(vertices):
    return (vertices / (2 * math.pi)) * math.sin(2 * math.pi / vertices)

print(f"{'steps/quadrant':>14} {'vertices':>9} {'area of the circle':>20} {'missing':>10}")
for steps in (1, 2, 4, 8, 16, 32, 64):
    vertices = 4 * steps
    share = fraction(vertices)
    mark = "  <- buffer_area" if steps == 8 else ("  <- buffer_point" if steps == 32 else "")
    print(f"{steps:>14} {vertices:>9} {share:>19.5%} {1 - share:>9.3%}{mark}")

print()
radius = 500
ideal = math.pi * radius ** 2
print(f"On a {radius} m radius, {ideal:.0f} m2 of ideal circle:")
for steps in (8, 32):
    missing = ideal * (1 - fraction(4 * steps))
    print(f"  {steps:>2} steps leaves {missing:7.1f} m2 outside the polygon ({missing / ideal:.3%})")
print()
print("Going from 8 to 32 recovers most of what is missing, and doubling again recovers little.")
print("Both of this project's settings sit either side of the knee, deliberately.")

## What this does not show

- That 500 m was the right distance. The widened boundary only set how far to fetch data. How far past the study area
  it should reach is a judgement about the data being fetched. A street network needs enough margin that routes near
  the edge are not cut short, which is why this widget existed at all.
- That the two buffers are the same shape. They are drawn on different maps, so only their areas are compared.